<a href="https://colab.research.google.com/github/simplyshree/SeqTrainer/blob/issue-3-all-model-baselines/notebooks/benchmarks_sg/dnabert_benchmark/dnabert2_v2_colab_hpc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DNABERT2 V2: Shared-Split Promoter Benchmark

This notebook improves the frozen DNABERT2 experiment while preserving the CNN-v2 comparison contract.

**Fixed across CNN-v2 and DNABERT2**

- exact train, validation, and held-out test CSV rows
- binary label meaning and seed policy rooted at `42`
- validation-only checkpoint, candidate, and probability-threshold selection
- MCC as the primary metric and AUPRC as the secondary metric
- accuracy, balanced accuracy, precision, recall/sensitivity, specificity, F1, AUROC, AUPRC, confusion counts, and predictions

**What V2 changes**

- trains the classifier head with shuffled mini-batches instead of one full-dataset update per epoch
- tests official 300 bp BPE length `70` plus controlled `104` and `128` ablations
- compares mean, CLS, and max pooling
- compares a linear head with a regularized MLP
- runs three seeds on Alpine and selects the candidate using validation results only

The frozen encoder stage is kept separate from later full fine-tuning. That prevents a large model-search exercise from being mistaken for one baseline.


## 1. Choose the execution profile

Use `colab` for a bounded one-seed screening run. Use `hpc` for the three-seed Alpine run. The same Python experiment implementation is used in both environments.


In [ ]:
PROFILE = "colab"  # "colab" here; Alpine uses --profile hpc in the sbatch file
REVISION = "issue-3-all-model-baselines"
SEED = 42


## 2. Colab only: install Conda

Choose **Runtime > Change runtime type > T4 GPU**. This cell restarts Colab. After reconnection, continue from Step 3 and do not rerun this cell.


In [ ]:
import os
if os.path.exists("/content"):
    !pip install -q condacolab
    import condacolab
    condacolab.install()
else:
    print("Not running in Colab; skip this cell.")


## 3. Clone SeqTrainer and create the pinned DNABERT2 environment

The older repository notebooks worked with Transformers `4.29.x` and without a usable Triton installation. Reusing that combination avoids the repeated `trans_b`, remote-config, and Transformers 5.x failures seen in newer default Colab environments.


In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/simplyshree/SeqTrainer.git"
REPO_DIR = Path("/content/SeqTrainer")
ENV_NAME = "dna"

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REVISION], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REVISION], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REVISION], check=True)
else:
    subprocess.run(["git", "clone", "--branch", REVISION, REPO_URL, str(REPO_DIR)], check=True)

envs = subprocess.check_output(["conda", "env", "list"], text=True)
if not any(line.split() and line.split()[0] == ENV_NAME for line in envs.splitlines()):
    subprocess.run(["conda", "create", "-y", "-n", ENV_NAME, "python=3.10", "pip"], check=True)

pip = ["conda", "run", "-n", ENV_NAME, "python", "-m", "pip"]
subprocess.run(pip + ["install", "-q", "torch==2.2.2", "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)
subprocess.run(pip + ["uninstall", "-y", "triton"], check=False)
subprocess.run(pip + [
    "install", "-q", "--force-reinstall",
    "transformers==4.29.2", "einops==0.6.1",
    "numpy==1.24.4", "pandas==2.0.3", "scikit-learn==1.3.2",
], check=True)
subprocess.run(pip + ["install", "-q", "-e", str(REPO_DIR)], check=True)

commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("Repository:", REPO_DIR)
print("Commit:", commit)


## 4. Verify GPU and model loading before the long run

This small preflight catches environment problems before embedding hundreds of thousands of sequences. DNABERT2 uses native BPE (`kmer=-1` in the official fine-tuning scripts); no k-mer conversion is added for CNN parity.


In [ ]:
PREFLIGHT = REPO_DIR / "check_dnabert2_v2.py"
PREFLIGHT.write_text(r'''
import torch
from seqtrainer.torch.dnabert2_benchmark import _last_hidden_state, _load_huggingface_dnabert2

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before continuing.")
device = torch.device("cuda")
tokenizer, model = _load_huggingface_dnabert2(
    "zhihan1996/DNABERT-2-117M",
    device=device,
    trust_remote_code=True,
    local_files_only=False,
    disable_flash_attention=True,
)
batch = tokenizer("ACGT" * 75, truncation=True, max_length=70, return_tensors="pt")
batch = {key: value.to(device) for key, value in batch.items()}
with torch.inference_mode():
    hidden = _last_hidden_state(model(**batch))
print("GPU:", torch.cuda.get_device_name(0))
print("Hidden state:", tuple(hidden.shape))
''', encoding="utf-8")
subprocess.run(["conda", "run", "--no-capture-output", "-n", ENV_NAME, "python", str(PREFLIGHT)], check=True)


## 5. Materialize the exact CNN split

No new split is generated. The bundled archive contains the same predefined rows used by CNN-v2.


In [ ]:
import zipfile

ARCHIVE = REPO_DIR / "data" / "data_DNABERT" / "promoter_classification_DNABERT.zip"
DATA_DIR = REPO_DIR / "data" / "promoter_classification"
DATA_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_FILES = {
    "train": "train_EP_DNA_BERT2_genomic_order.csv",
    "validation": "eval_EP_DNA_BERT2_genomic_order.csv",
    "test": "test_EP_DNA_BERT2_genomic_order.csv",
}

missing = [name for name in SPLIT_FILES.values() if not (DATA_DIR / name).exists()]
if missing:
    if not ARCHIVE.exists():
        raise FileNotFoundError(ARCHIVE)
    with zipfile.ZipFile(ARCHIVE) as archive:
        for name in missing:
            with archive.open(name) as source, (DATA_DIR / name).open("wb") as target:
                target.write(source.read())

for split, name in SPLIT_FILES.items():
    print(split, DATA_DIR / name)


## 6. Audit the fixed data before training

The current data is nearly balanced, so V2 does not force class weighting. If a future dataset is imbalanced, the imbalance policy must be estimated from training labels only.


In [ ]:
import pandas as pd

for split, name in SPLIT_FILES.items():
    frame = pd.read_csv(DATA_DIR / name)
    print("\n", split, "rows:", len(frame))
    print(frame["label"].value_counts().sort_index())
    print("positive fraction:", frame["label"].astype(int).mean())


## 7. Run DNABERT2 V2

The Colab profile screens token lengths `70/104`, three pooling methods, two heads, and seed `42`. Cached embeddings make head trials inexpensive, but extracting the full dataset can still exceed a free Colab session. Use Alpine for the complete three-seed profile.


In [ ]:
OUTPUT_DIR = REPO_DIR / "outputs" / "benchmarks" / "dnabert2_v2_colab"
EXPERIMENT = REPO_DIR / "notebooks" / "benchmarks_sg" / "dnabert_benchmark" / "dnabert2_v2_experiment.py"

command = [
    "conda", "run", "--no-capture-output", "-n", ENV_NAME, "python", str(EXPERIMENT),
    "--profile", PROFILE,
    "--repo-dir", str(REPO_DIR),
    "--data-dir", str(DATA_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--resume",
]
print(" ".join(command))
subprocess.run(command, check=True)


## 8. Inspect validation selection and final held-out test metrics

Candidate ranking uses mean validation MCC first and mean validation AUPRC second. Test metrics below are final reporting only; they are never used to choose the candidate or threshold.


In [ ]:
from IPython.display import display

candidate_summary = pd.read_csv(OUTPUT_DIR / "validation_candidate_summary.csv")
test_metrics = pd.read_csv(OUTPUT_DIR / "metrics.csv")
test_summary = pd.read_csv(OUTPUT_DIR / "selected_test_summary.csv")
display(candidate_summary.head(12))
display(test_metrics)
display(test_summary)


## 9. Plot the validation ranking and selected training history


In [ ]:
import matplotlib.pyplot as plt

top = candidate_summary.head(12).copy()
top["candidate"] = (
    "L" + top["token_length"].astype(str)
    + " | " + top["pooling"]
    + " | " + top["head"]
    + " | lr=" + top["learning_rate"].map(lambda value: f"{value:.0e}")
)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].barh(top["candidate"][::-1], top["mean_validation_mcc"][::-1])
axes[0].set_title("Validation MCC candidate ranking")
axes[0].set_xlabel("Mean validation MCC")

history = pd.read_csv(OUTPUT_DIR / "history.csv")
for seed, group in history.groupby("seed"):
    axes[1].plot(group["epoch"], group["validation_mcc"], label=f"seed {seed}")
axes[1].set_title("Selected candidate validation MCC")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MCC")
axes[1].legend()
plt.tight_layout()
plt.show()


## 10. Compare with the recorded CNN-v2 target

The previously recorded CNN-v2 held-out test result was MCC `0.220884` and AUPRC `0.645976`. DNABERT2-v2 should be judged primarily against those two values, not accuracy alone.


In [ ]:
CNN_V2_TEST_MCC = 0.220884
CNN_V2_TEST_AUPRC = 0.645976
mean_values = dict(zip(test_summary["metric"], test_summary["mean"]))

comparison = pd.DataFrame([
    {"model": "CNN-v2", "test_mcc": CNN_V2_TEST_MCC, "test_auprc": CNN_V2_TEST_AUPRC},
    {
        "model": "DNABERT2-v2 frozen",
        "test_mcc": mean_values["mcc"],
        "test_auprc": mean_values["auprc"],
    },
])
display(comparison)


## 11. Alpine HPC full run

Copy the repository to Alpine, update `<YOUR_ALPINE_ACCOUNT>` in `alpine_dnabert2_v2.sbatch`, then submit:

```bash
cd /path/to/SeqTrainer
sbatch notebooks/benchmarks_sg/dnabert_benchmark/alpine_dnabert2_v2.sbatch
```

The Alpine profile uses one A100 GPU, token lengths `70/104/128`, both heads, both head learning rates, and seeds `42/43/44`. Resume support reuses completed embedding caches after interruption.

After V2, the next scientifically distinct experiment is encoder fine-tuning at approximately `1e-5` to `3e-5`, effective batch size `32`, and the same validation-MCC/test-holdout policy. It should be reported separately from this frozen baseline.
